In [17]:
import plotly.express as px
import requests
import pandas as pd

# 1. Fetch municipality GeoJSON from DAWA
geojson = requests.get(
    "https://api.dataforsyningen.dk/kommuner?format=geojson"
).json()

# Each feature has properties: "kode" (4-digit code), "navn" (name)
# Extract names and codes for a base dataframe
municipalities = [
    {"regionskode": f["properties"]["regionskode"], "navn": f["properties"]["navn"]}
    for f in geojson["features"]
]
df = pd.DataFrame(municipalities)


In [ ]:
#geojson.keys() # ['type', 'crs', 'features']
geojson_data = geojson['features']
#geojson_data[0].keys() # ['type', 'geometry', 'properties', 'bbox']
geojson_data[0]['properties'] # dict_keys(['kod', 'navn', 'regionskode', 'regionsnavn'])
for region in geojson['features']:
    print(region['properties']['regionskode'], region['properties']['navn'])

In [ ]:
# 2. (Optional) Merge your own data — e.g. employee counts
# df = df.merge(your_df, on="navn", how="left")
# For demo, just use a dummy value
df["value"] = 1

# 3. Plot
fig = px.choropleth_map(
    df,
    geojson=geojson,
    locations="regionskode",                          # column in df to match GeoJSON
    featureidkey="properties.regionskode",            # key inside GeoJSON features
    color="value",                             # column to color by
    hover_name="navn",
    # map_style="carto-positron",
    center={"lat": 56.0, "lon": 10.5},
    zoom=5.5,
    opacity=0.6,
    color_continuous_scale="Blues",
    range_color=(0.5, 1.5),
    title="Danish Municipalities"
)
fig.update_layout(margin={"r":0,"t":40,"l":0,"b":0})
fig.show()

In [ ]:
import folium
import requests

# 1. Fetch GeoJSON
geojson = requests.get(
    "https://api.dataforsyningen.dk/kommuner?format=geojson"
).json()

# 2. Create base map centered on Denmark
m = folium.Map(
    location=[56.0, 10.5],
    zoom_start=7,
    tiles="CartoDB positron"
)

# 3. Add municipality boundaries with popups
folium.GeoJson(
    geojson,
    name="Municipalities",
    style_function=lambda feature: {
        "fillColor": "#4a90d9",
        "color": "#1a1a2e",
        "weight": 1,
        "fillOpacity": 0.4,
    },
    highlight_function=lambda feature: {
        "fillColor": "#f0a500",
        "fillOpacity": 0.7,
        "weight": 2,
    },
    tooltip=folium.GeoJsonTooltip(
        fields=["navn", "kode"],
        aliases=["Municipality:", "Code:"],
        localize=True,
    ),
).add_to(m)

folium.LayerControl().add_to(m)
m.save("municipalities.html")   # open in browser
m                                # renders inline in Jupyter